In [9]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential

In [ ]:
# Opening a zip file
from zipfile import ZipFile
with ZipFile('Data/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('archive')

In [11]:
# Data paths
yes_data = 'Data/archive/brain_tumor_dataset/yes'
no_data = 'Data/archive/brain_tumor_dataset/no'

In [12]:
# Inspect number of images in each category
import os
num_yes = len(os.listdir(yes_data))
num_no = len(os.listdir(no_data))
print(f"Number of 'yes' images: {num_yes}")
print(f"Number of 'no' images: {num_no}")

Number of 'yes' images: 155
Number of 'no' images: 98


In [ ]:
import os, random, shutil

YES_DIR = r"Data/archive/brain_tumor_dataset/yes"      
NO_DIR  = r"Data/archive/brain_tumor_dataset/no"     
OUT_DIR = r"Data/archive/brain_tumor_just_split"   

random.seed(42)
os.makedirs(os.path.join(OUT_DIR, "yes"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "no"), exist_ok=True)

# Get all images
yes_images = [f for f in os.listdir(YES_DIR) if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff','.gif'))]
no_images  = [f for f in os.listdir(NO_DIR)  if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff','.gif'))]

# Find equal count
n = min(len(yes_images), len(no_images))
print(f"Using {n} images from each class.")

# Randomly sample equal subsets
yes_subset = random.sample(yes_images, n)
no_subset  = random.sample(no_images,  n)

# Copy to new balanced folder
for f in yes_subset:
    shutil.copy2(os.path.join(YES_DIR, f), os.path.join(OUT_DIR, "yes", f))
for f in no_subset:
    shutil.copy2(os.path.join(NO_DIR, f), os.path.join(OUT_DIR, "no", f))

print("Balanced dataset created here:", OUT_DIR)


Using 98 images from each class.
Balanced dataset created here: Data/archive/brain_tumor_just_split


In [15]:
yes_data = 'Data/archive/brain_tumor_just_split/yes'
no_data = 'Data/archive/brain_tumor_just_split/no'


num_yes = len(os.listdir(yes_data))
num_no = len(os.listdir(no_data))
print(f"Number of 'yes' images: {num_yes}")
print(f"Number of 'no' images: {num_no}")

Number of 'yes' images: 98
Number of 'no' images: 98


In [ ]:
import math

SRC_DIR = r"Data/archive/brain_tumor_just_split"   # contains balanced_data/yes and balanced_data/no
OUT_DIR = r"Data/archive/brain_tumor_prepped"
SPLIT = (0.70, 0.15, 0.15)  # train, val, test ratios

random.seed(42)
CLASSES = ["yes", "no"]
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff")

# make destination folders
for split in ["train", "val", "test"]:
    for cls in CLASSES:
        os.makedirs(os.path.join(OUT_DIR, split, cls), exist_ok=True)

# split and copy
for cls in CLASSES:
    src_folder = os.path.join(SRC_DIR, cls)
    files = [f for f in os.listdir(src_folder)
             if f.lower().endswith(EXTS) and os.path.isfile(os.path.join(src_folder, f))]
    random.shuffle(files)

    n = len(files)
    n_train = math.floor(SPLIT[0] * n)
    n_val   = math.floor(SPLIT[1] * n)

    train_files = files[:n_train]
    val_files   = files[n_train:n_train + n_val]
    test_files  = files[n_train + n_val:]

    for f in train_files:
        shutil.copy2(os.path.join(src_folder, f), os.path.join(OUT_DIR, "train", cls, f))
    for f in val_files:
        shutil.copy2(os.path.join(src_folder, f), os.path.join(OUT_DIR, "val", cls, f))
    for f in test_files:
        shutil.copy2(os.path.join(src_folder, f), os.path.join(OUT_DIR, "test", cls, f))

    print(f"{cls}: total={n} → train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

print("Done! Your folders are ready at:", OUT_DIR)

yes: total=98 → train=68, val=14, test=16
no: total=98 → train=68, val=14, test=16
Done! Your folders are ready at: Data/archive/brain_tumor_prepped


In [105]:
# Preparing the data
train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.2,
                                   rotation_range=20,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   fill_mode="nearest"
                                )

train_set = train_datagen.flow_from_directory('Data/archive/brain_tumor_prepped/train',
                                                    target_size=(128, 128),
                                                    batch_size=32,
                                                    class_mode='binary',
                                                    shuffle=True
                                                    )

Found 136 images belonging to 2 classes.


In [106]:
# 
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory('Data/archive/brain_tumor_prepped/test',
                                                  target_size=(128, 128),
                                                  batch_size=32,
                                                  class_mode='binary',
                                                  shuffle=False)

Found 32 images belonging to 2 classes.


In [107]:
# Initialize the CNN
cnn = tf.keras.models.Sequential()
# Step 1 - Convolution
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[128, 128, 3]))
# Step 2 - Pooling
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))
# Adding a second convolutional layer
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))
# Step 3 - Flattening
cnn.add(tf.keras.layers.Flatten())
# Step 4 - Full Connection
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
# Step 5 - Output Layer
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

c:\Users\User\anaconda3\envs\learn-env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [108]:
# Compiling the CNN
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [109]:
# Training the CNN
cnn.fit(x=train_set, validation_data=test_set, epochs=30)


c:\Users\User\anaconda3\envs\learn-env\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 933ms/step - accuracy: 0.5809 - loss: 0.6618 - val_accuracy: 0.7812 - val_loss: 0.5632
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 604ms/step - accuracy: 0.7426 - loss: 0.5721 - val_accuracy: 0.7500 - val_loss: 0.5096
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 602ms/step - accuracy: 0.6103 - loss: 0.7400 - val_accuracy: 0.7812 - val_loss: 0.5776
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 611ms/step - accuracy: 0.6691 - loss: 0.5923 - val_accuracy: 0.5938 - val_loss: 0.5975
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 652ms/step - accuracy: 0.6471 - loss: 0.6067 - val_accuracy: 0.8125 - val_loss: 0.4821
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 564ms/step - accuracy: 0.6912 - loss: 0.5825 - val_accuracy: 0.8125 - val_loss: 0.4455
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 549ms/step - accuracy: 0.6765 - loss: 0.6231 - val_accuracy: 0.8125 - val_loss: 0.5367
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 535ms/step - accuracy: 0.7500 - loss: 0.5433 - val_accuracy: 0.8438 - val_loss:

In [110]:
cnn.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_14 (Conv2D)              │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 61, 61, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_12 (Flatten)            │ (None, 28800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 128)            │     3,686,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,090,405 (42.31 MB)

 Trainable params: 3,696,801 (14.10 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,393,604 (28.20 MB)

In [111]:
# evaluation
test_loss, test_acc = cnn.evaluate(test_set)
print('\nTest accuracy:', test_acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.8125 - loss: 0.3731

Test accuracy: 0.8125


In [112]:
# confusion matrix and classification report
from sklearn.metrics import classification_report, confusion_matrix
y_pred = cnn.predict(test_set)
y_pred = (y_pred > 0.5).astype(int)
print(confusion_matrix(test_set.classes, y_pred))
print(classification_report(test_set.classes, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step
[[15  1]
 [ 5 11]]
              precision    recall  f1-score   support

           0       0.75      0.94      0.83        16
           1       0.92      0.69      0.79        16

    accuracy                           0.81        32
   macro avg       0.83      0.81      0.81        32
weighted avg       0.83      0.81      0.81        32



In [113]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model

In [114]:
pretrained_model = ResNet50(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
for layer in pretrained_model.layers:
    layer.trainable = False
x = pretrained_model.output
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dense(1, activation='sigmoid')(x)
resnet_cnn = Model(inputs=pretrained_model.input, outputs=x)

In [115]:
resnet_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
resnet_cnn.fit(x=train_set, validation_data=test_set, epochs=30)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.4853 - loss: 1.4025 - val_accuracy: 0.5000 - val_loss: 0.8485
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.5000 - loss: 0.8583 - val_accuracy: 0.5000 - val_loss: 0.7084
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.5368 - loss: 0.6741 - val_accuracy: 0.5000 - val_loss: 0.7046
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.5000 - loss: 0.6951 - val_accuracy: 0.5000 - val_loss: 0.7014
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.5809 - loss: 0.6650 - val_accuracy: 0.5938 - val_loss: 0.6637
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.6838 - loss: 0.6363 - val_accuracy: 0.6562 - val_loss: 0.6524
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.6471 - loss: 0.6291 - val_accuracy: 0.6250 - val_loss: 0.6465
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.6618 - loss: 0.6251 - val_accuracy: 0.6250 - val_loss: 0.6548
Epoch 9/30
5/5 

In [116]:
# Evaluation
test_loss, test_acc = resnet_cnn.evaluate(test_set)
print('\nTest accuracy with ResNet50:', test_acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7812 - loss: 0.4955

Test accuracy with ResNet50: 0.78125


In [117]:
# confusion matrix and classification report
from sklearn.metrics import classification_report, confusion_matrix
y_pred = resnet_cnn.predict(test_set)
y_pred = (y_pred > 0.5).astype(int)
print(confusion_matrix(test_set.classes, y_pred))
print(classification_report(test_set.classes, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
[[11  5]
 [ 2 14]]
              precision    recall  f1-score   support

           0       0.85      0.69      0.76        16
           1       0.74      0.88      0.80        16

    accuracy                           0.78        32
   macro avg       0.79      0.78      0.78        32
weighted avg       0.79      0.78      0.78        32

